## Multi Agent Workflow

In [1]:
from typing_extensions import TypedDict

class NewsroomState(TypedDict):
    stories: dict        

### Agent 1: Trend Fetcher


In [2]:
from agents.agent1 import fetch_trends

In [3]:
import re

def make_story_id(title: str) -> str:
    # input: melodi India & Italy Diplomatic Relations
    # output:melodi india-&-italy-diplomatic-relations(lowecase ouput)
    return re.sub(r'[^a-z0-9]+', '-', title.lower()).strip('-')[:50]


In [4]:
print(make_story_id("Melody India Italy Diplomatic Relations"))

melody-india-italy-diplomatic-relations


In [5]:
# AGENT 1 NODE — creates stories dict
def trend_hunter_node(state: NewsroomState) -> dict:
    trends = fetch_trends()
    stories = {}
    for t in trends:
        #print(t)
        sid = make_story_id(t["title"])
        stories[sid] = t             
    return {"stories": stories}

In [6]:
call = trend_hunter_node(NewsroomState(stories={}))
print(call)

{'stories': {'vulnerability-reports-are-not-special-anymore': {'title': 'Vulnerability reports are not special anymore', 'url': 'https://words.filippo.io/vuln-reports/', 'source': 'hackernews', 'category': 'technology', 'engagement': 398, 'velocity': 57.5}, 'futo-swipe-a-new-swipe-typing-model': {'title': 'FUTO Swipe – A new swipe typing model', 'url': 'https://swipe.futo.tech/', 'source': 'hackernews', 'category': 'technology', 'engagement': 695, 'velocity': 56.4}, 'jerry-s-map': {'title': "Jerry's Map", 'url': 'http://www.jerrysmap.com/the-map', 'source': 'hackernews', 'category': 'technology', 'engagement': 514, 'velocity': 39.3}, 'in-memory-of-the-man-who-put-red-and-green-squiggl': {'title': 'In memory of the man who put red and green squiggles under words', 'url': 'https://devblogs.microsoft.com/oldnewthing/20260622-00/?p=112451', 'source': 'hackernews', 'category': 'technology', 'engagement': 427, 'velocity': 32.7}, 'raspberry-pi-pico-w-as-usb-wi-fi-adapter': {'title': 'Raspberr

In [7]:
print(f"Stories created: {len(call['stories'])}")
print("IDs:", list(call['stories'].keys()))

Stories created: 8
IDs: ['vulnerability-reports-are-not-special-anymore', 'futo-swipe-a-new-swipe-typing-model', 'jerry-s-map', 'in-memory-of-the-man-who-put-red-and-green-squiggl', 'raspberry-pi-pico-w-as-usb-wi-fi-adapter', 'qwen-agentworld-language-world-models-for-general-', 'show-hn-an-ascii-3d-rendering-engine', 'why-eval-startups-fail-2025']


In [8]:
call['stories']

{'vulnerability-reports-are-not-special-anymore': {'title': 'Vulnerability reports are not special anymore',
  'url': 'https://words.filippo.io/vuln-reports/',
  'source': 'hackernews',
  'category': 'technology',
  'engagement': 398,
  'velocity': 57.5},
 'futo-swipe-a-new-swipe-typing-model': {'title': 'FUTO Swipe – A new swipe typing model',
  'url': 'https://swipe.futo.tech/',
  'source': 'hackernews',
  'category': 'technology',
  'engagement': 695,
  'velocity': 56.4},
 'jerry-s-map': {'title': "Jerry's Map",
  'url': 'http://www.jerrysmap.com/the-map',
  'source': 'hackernews',
  'category': 'technology',
  'engagement': 514,
  'velocity': 39.3},
 'in-memory-of-the-man-who-put-red-and-green-squiggl': {'title': 'In memory of the man who put red and green squiggles under words',
  'url': 'https://devblogs.microsoft.com/oldnewthing/20260622-00/?p=112451',
  'source': 'hackernews',
  'category': 'technology',
  'engagement': 427,
  'velocity': 32.7},
 'raspberry-pi-pico-w-as-usb-wi-

### Agent 2: Content Fetcher

In [9]:
import ollama, subprocess, time

try:
    ollama.list()
    print("✅ Ollama already running")
except Exception:
    print("⏳ Starting Ollama...")
    subprocess.Popen(["ollama", "serve"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    try:
        ollama.list()
        print("✅ Ollama started")
    except Exception:
        print("❌ Run `ollama serve` in a terminal manually")

✅ Ollama already running


In [10]:
from agents.agent2 import fetch_url_content,fetch_trend_background

In [11]:
# AGENT 2 NODE — APPENDS content + background to EACH story in place
def context_researcher_node(state: NewsroomState) -> dict:
    stories = state["stories"]
    for sid, story in stories.items():
        print("Agent 2 Starts For :",sid)
        story["content"]    = fetch_url_content(story["url"])      # ← appends key
        story["background"] = fetch_trend_background(story["title"],story["content"]) # ← appends key
        print(f"  Agent 2 Ends for: {story['title']}")
        print("="*80)
    return {"stories": stories}

In [12]:
call2 = context_researcher_node(call)


Agent 2 Starts For : vulnerability-reports-are-not-special-anymore
Jina Success ✅  In Loading Content:10269 characters
  [background] gathering snippets for: Vulnerability reports are not special anymore
[ddg] error for 'Vulnerability reports are not special anymore': No results found.
  [bg] DDG returned 0 snippets
  [bg] Wikipedia search returns: 1334 chars
  [background] 1 snippets, synthesizing (content-anchored)...
Result After Background Syntezing is : INSUFFICIENT DATA
  [bg] background: EMPTY-No Data
  Agent 2 Ends for: Vulnerability reports are not special anymore
Agent 2 Starts For : futo-swipe-a-new-swipe-typing-model
Trafiltura Success ✅  In Loading Content :686 characters
  [background] gathering snippets for: FUTO Swipe – A new swipe typing model
[ddg] error for 'FUTO Swipe – A new swipe typing model': No results found.
  [bg] DDG returned 0 snippets
  [bg] Wikipedia search returns: empty (skipped)
  [background] 0 snippets, synthesizing (content-anchored)...
  [bg] backg

In [13]:
print(call2['stories'].keys())

dict_keys(['vulnerability-reports-are-not-special-anymore', 'futo-swipe-a-new-swipe-typing-model', 'jerry-s-map', 'in-memory-of-the-man-who-put-red-and-green-squiggl', 'raspberry-pi-pico-w-as-usb-wi-fi-adapter', 'qwen-agentworld-language-world-models-for-general-', 'show-hn-an-ascii-3d-rendering-engine', 'why-eval-startups-fail-2025'])


In [14]:
for sid in call2['stories'].keys():
    story = call2['stories'][sid]

    print("TITLE:     ", story['title'])
    print("URL:       ", story['url'])
    print("VELOCITY:  ", story['velocity'])
    print("\nCONTENT:\n", story['content'])
    print("\nBACKGROUND:\n", story['background'] if story['background'] else "(empty)")

    print("="*100)

TITLE:      Vulnerability reports are not special anymore
URL:        https://words.filippo.io/vuln-reports/
VELOCITY:   57.5

CONTENT:
 23 Jun 2026
A requirement for staying sane while working in public as an open source maintainer is realizing that every issue, PR, and piece of feedback is a present, not an obligation. You can accept it, ignore it, and use it partially or not at all.

Except…

For years, as lead of the Go Security team at the time,[1](https://words.filippo.io/vuln-reports/#fn:lead) I’ve told new team members that it doesn’t apply to vulnerability reports. No, vulnerability reports are special. Security researchers are doing us a favor by reporting things confidentially instead of doing full disclosure, so we owe them something, which is not true of regular issues opened on the issue tracker.[2](https://words.filippo.io/vuln-reports/#fn:coc)

Different projects have different policies, but the general expectations are responsiveness and attribution. We’re supposed to 

In [15]:
# Manual Testing Required

# TODO: [TEST-4] fresh story batch (different time of day)
#        - news-heavy stories get good backgrounds?
#        - tier fallback fires on different sites?
#        - new junk patterns looks_like_real_content misses?
# TODO: [TEST-5] verify 2 empty-with-snippets cases
#        - swift-package-index + steam-machine-opinion
#        - is it correct INSUFFICIENT DATA or over-caution?
# TODO: [TEST-6] (optional) known news URL (Reuters/TechCrunch)
#        - background quality GOOD when snippets relevant?

In [16]:
# Automated Testing 

# tests/test_agent2.py
from agents.agent2 import looks_like_real_content, clean_jina

def test_rejects_language_menu():
    menu = "sign in\nlanguage\n简体中文\n日本語\n한국어"
    assert looks_like_real_content(menu) == False

def test_accepts_real_article():
    article = ("Valve announced Steam Machine pricing today after months of delays.\n"
               "The base model starts at $1,049 with component costs driving the price.\n"
               "Pre-orders open June 8th ahead of the June 29th release date.\n"
               "The semi-custom AMD platform faced supply chain challenges this year.")
    assert looks_like_real_content(article) == True

def test_clean_jina_strips_header():
    text = "Title: X\n\nMarkdown Content:\n" + "real body " * 50
    assert not clean_jina(text).startswith("Title:")


# at the bottom of the cell, call them:
test_rejects_language_menu()
test_accepts_real_article()
test_clean_jina_strips_header()
print("✅ all tests passed")

✅ all tests passed
